# Mechanism Tutorial 02 — RBD and Memory X_t → H_{t+1} (protocol_h_rbd_memory)

**Continues 01's conceptual model** — same `make_shared_column(N=200, ...)` helper, **no restart**. Builds on existence→expression to show **state dynamics, memory, recovery, continuation, and timescales**.

We show:
- H initialized away from star, autonomous F1 recovery `τ_K·dH_K/dt = 1−H_K`
- perturbation formation → distributed X change → fading with H
- exact continuation (segmented vs continuous) with `delay_state` + `continuation_step_offset` (H2 contract)
- timescale sweep (fast vs slow τ_H)
- protocol_h_rbd_memory framing (F0/F1/F2, β_H gain, heterogeneous delays are the H4 inference target; here we demo the mechanism)

**API:** `simulate_edge_recurrent_izhikevich_dynamic_h_k_recovery`, `simulate_edge_recurrent_izhikevich_rbd`, `simulate_edge_recurrent_izhikevich_owned_h_k_delayed` — existing emitters only.

## Notebook grammar

setup (reuse 01's builder) → configured RBD → realized H arrays → effective traces → continuation exactness → timescale comparison → configured→realized→effective

## Scope & lineage

- 01 taught **existence vs expression** (Γ_H = I vs active).
- 02 teaches **memory**: H carries history across steps (X_t → H_{t+1}) and relaxes with its own timescale.
- 03 will close the loop H→W via HDP.

All three share **one circuit** (200-neuron V1 column, canonical 1000n reference). Changing `N` rescales without redefining the mechanism — see `jaxfne.hdp_network.HDPColumnConfig` for the scalable-pattern.

## Colab Installation

In [ ]:
# Colab / local install: use checkout when present, otherwise pip from main.
import importlib.util, subprocess, sys
from pathlib import Path
for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "jaxfne").is_dir() and (_candidate / "pyproject.toml").exists():
        sys.path.insert(0, str(_candidate))
        break
if importlib.util.find_spec("jaxfne") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "jaxfne[viz,opt] @ git+https://github.com/HNXJ/jaxfne.git@main"])

## Imports

In [ ]:
import os, json, hashlib
import numpy as np
import numpy as _np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import jaxfne as jtfne
print(f"jaxfne {jtfne.__version__}")
from jaxfne.emitters import (simulate_edge_recurrent_izhikevich_dynamic_h_k_recovery, simulate_edge_recurrent_izhikevich_rbd, simulate_edge_recurrent_izhikevich_owned_h_k_delayed)

## Reuse 01's Shared Conceptual Model (configured → realized)

**Same function, same seed, same N** — not a new circuit. 02 extends the *dynamics*, not the declaration.

In [ ]:
# Shared conceptual model — same builder reused in 01 -> 02 -> 03 (not restarted).
# Configured -> realized -> effective. Canonical 200-neuron column for speed
# (swap N=1000 for full canonical run; behaviour scales).

def make_shared_column(n=200, seed=0, dt_ms=0.5, duration_ms=300.0):
    """Build the progressive mechanism column (configured -> realized)."""
    cfg = (
        jtfne.Configuration()
        .runtime(seed=int(seed), duration_ms=float(duration_ms), dt_ms=float(dt_ms), dtype="float32", recurrent_backend="edge_list")
        .areas(["V1"])
        .column("V1", layers=["L2/3", "L4", "L5", "L6"], n=int(n))
        .cell_types({"E": 0.75, "PV": 0.10, "SST": 0.08, "VIP": 0.07})
        .uniform3d(radius_mm=0.25, height_mm=1.6)
        .connectivity(within_area="all_to_all_uniform_random", within_gain=0.35, edge_seed=int(seed))
        .set_emitter("izhikevich", "cortical_eig")
        .probes(["spikes", "V_m", "source"], n_contacts=8)
        .field(domain="laminar_column", conductivity="proxy", boundary="mean_zero_neumann", gauge="mean_zero")
    )
    return jtfne.construct(cfg)

N = 200          # canonical 1000n is the reference; 200 used for CI speed
DT_MS = 0.5
DURATION_MS = 300.0
SEED = 7
model = make_shared_column(n=N, seed=SEED, dt_ms=DT_MS, duration_ms=DURATION_MS)
print("realized:", model.summary())
print("neuron_table head:", model.neuron_table()[:2])
print("edges:", model.params["edge_list"].n_edges)
# Configured (what we declared) vs realized (what construct() built) vs effective (what simulate() produces)
import jaxfne.util as _util
try:
    print(_util.canonical_compact_summary(model))
except Exception as e:
    print("compact summary unavailable:", e)

# 02 lengthens the window to show recovery tails
DURATION_MS_02 = 400.0
n_steps_02 = int(round(DURATION_MS_02 / DT_MS))
print(f"02 reuses 01's column: N={N}, edges={model.params['edge_list'].n_edges}, extended window {DURATION_MS_02} ms ({n_steps_02} steps)")

## Perturbation & autonomous recovery (D2a F1)

Freeze `τ_K·dH_K/dt = 1−H_K` (F1 Euler, no I_rel coupling, κ_K=0). Perturb H_K far from star (e.g. H_K⁰=2.0 on all neurons; single-neuron shown earlier gave X signal at H1c, here we show **temporal fading**).

In [ ]:
params = model.params["emitter"]; edges = model.params["edge_list"]
n = params.n_neurons
key = jax.random.PRNGKey(SEED)

h_k0_pert = jnp.ones(n, dtype=jnp.float32) * 2.0   # uniform offset to make decay visible in mean
tau_k_ms = 50.0   # fast enough to see within 400 ms
V_dyn, S_dyn, src_dyn, info_dyn = simulate_edge_recurrent_izhikevich_dynamic_h_k_recovery(
    params, edges, n_steps_02, DT_MS, key, h_k0=h_k0_pert, tau_k_ms=tau_k_ms, noise_scale=0.0)

H_trace = _np.asarray(info_dyn["H_K_trace"])  # (steps, n)
print(f"perturbed H_K[0] at t=0: {float(H_trace[0,0]):.3f}, t=end: {float(H_trace[-1,0]):.3f}, star=1.0")
# Discrete contract H^{n+1}=H^n+dt*(1-H^n)/tau ; analytic envelope (1 + (H0-1)*exp(-t/tau))
import math
pred_end = 1.0 + (2.0-1.0)*math.exp(-DURATION_MS_02/tau_k_ms)
print(f"analytic envelope at t={DURATION_MS_02} ms: {pred_end:.3f} (Euler approaches it; exact match not required for scaffold gate)")
# Effective: X reflects transient while H>1 (recovery window)
rate_dyn = float(_np.asarray(S_dyn).mean()*1000.0/DT_MS)
print(f"effective rate (perturbed run): {rate_dyn:.2f} Hz — transiently elevated while H>1")


## Protocol H RBD: F0 / F1 / F2 and H→X gain β_H

F0 null (Ḣ=0, H≡1) isolates delay/activity memory; F1 linear and F2 inverse-state families share equilibrium and Jacobian at H*=1 but differ away from it. Here we show **F1 with β_H>0** so the recovery transient is **expressed** (as in 01).

In [ ]:
# RBD run with explicit initial H≠1 to see F1 relaxation expressed via β_H
H0_rbd = jnp.ones(n, dtype=jnp.float32).at[0].set(2.5)
# Need full init_state for RBD (v,u,prev_spikes,syn_state + H). Build from model defaults.
init_rbd = {"H": H0_rbd, "v": params.v0, "u": params.u0, "prev_spikes": jnp.zeros(n, dtype=jnp.float32), "syn_state": jnp.zeros(edges.n_edges, dtype=jnp.float32)}
Vr, Sr, srcr, infor = simulate_edge_recurrent_izhikevich_rbd(
    params, edges, n_steps_02, DT_MS, key, rbd_family="f1", tau_h_ms=80.0, kappa_h=0.0, beta_h=0.2, noise_scale=0.0, init_state=init_rbd)

Hr = _np.asarray(infor["H_trace"])
print(f"RBD F1 β_H=0.2: H0[0]={float(Hr[0,0]):.3f} → H_end[0]={float(Hr[-1,0]):.3f}")
# Compare to F0 null (H≡1, β irrelevant) — should show no H transient
_, S0, _, info0 = simulate_edge_recurrent_izhikevich_rbd(params, edges, n_steps_02, DT_MS, key, rbd_family="f0", tau_h_ms=80.0, noise_scale=0.0)
print(f"F0 null H (first step): {float(_np.asarray(info0['H_trace'])[0,0]):.2f} ≡1 (by construction)")
# Effective X difference
diff_rbd = int(_np.abs(_np.asarray(Sr)-_np.asarray(S0)).sum())
print(f"effective X: F1-perturbed vs F0-null Δspikes={diff_rbd} (>0 shows H→X pathway)")


## Recovery & timescales (protocol task: different τ)

Same perturbation, two τ values. Fast τ recovers in tens of ms; slow τ preserves the transient — the **timescale is the memory knob** (without yet invoking W).

In [ ]:
def _run_tau(tau):
    return simulate_edge_recurrent_izhikevich_dynamic_h_k_recovery(params, edges, n_steps_02, DT_MS, key, h_k0=h_k0_pert, tau_k_ms=float(tau), noise_scale=0.0)

_, _, _, info_fast = _run_tau(30.0)
_, _, _, info_slow = _run_tau(200.0)
Hf = _np.asarray(info_fast["H_K_trace"])[:,0]
Hs = _np.asarray(info_slow["H_K_trace"])[:,0]
t = _np.arange(n_steps_02)*DT_MS
# Check monotonic decay toward 1 and ordering fast<slow after e.g. 100 ms
idx_100 = int(100.0/DT_MS)
print(f"τ=30 ms at 100 ms: H={float(Hf[idx_100]):.3f}; τ=200 ms at 100 ms: H={float(Hs[idx_100]):.3f} (slow retains more)")
assert float(Hf[idx_100]) < float(Hs[idx_100]), "fast tau should have decayed more by 100 ms"
# Plot
fig, ax = plt.subplots(figsize=(8,3))
ax.plot(t, Hf, label="τ=30 ms (fast, fading)")
ax.plot(t, Hs, label="τ=200 ms (slow, memory)")
ax.axhline(1.0, color="k", ls="--", lw=0.8, label="H*=1")
ax.set(xlabel="time (ms)", ylabel="H_K[0]"); ax.set_title("RBD recovery timescales (protocol task: timescale sweep)"); ax.legend(); ax.grid(alpha=0.2)
plt.close(fig)
fig


## Continuation exactness (H2: delay_state + continuation_step_offset)

A continuous `T1+T2` run must equal two back-to-back segments with carry `(v,u,prev_spikes,syn_state,delay_state,H,continuation_step_offset)` when noise is matched (bit-exact float32 at noise_scale=0 per `tests/test_protocol_h_rbd_h2.py`). We demo on the **owned H_K + delay** kernel so the delay buffer is part of the proof.

In [ ]:
# Give the column heterogeneous delays so continuation must carry delay_state
import numpy as _np_host
edges_delayed = jtfne.emitters.edge_list_with_delay_ms(edges, delay_ms=_np_host.random.RandomState(SEED).randint(0,3,size=edges.n_edges).astype(float)*DT_MS, dt_ms=DT_MS)
print(f"heterogeneous delays: max {int(_np.max(_np.asarray(edges_delayed.delay_steps)))} steps, nonzero {int(_np.count_nonzero(_np.asarray(edges_delayed.delay_steps)))}")

# Continuous T=400 steps (noise_scale=0 -> deterministic, PRNG irrelevant for value but segment handling differs at low level)
n_cont = 400
key_c = jax.random.PRNGKey(SEED)
h0c = jnp.ones(n, dtype=jnp.float32)*1.6
owner_c = jnp.ones(n, dtype=jnp.float32)
V_full, S_full, src_full, st_full = simulate_edge_recurrent_izhikevich_owned_h_k_delayed(
    params, edges_delayed, n_cont, DT_MS, key_c, h_k0=h0c, owner_mask=owner_c, dynamic=True, tau_k_ms=80.0, gamma_h_enabled=True, noise_scale=0.0)

# Segmented 200+200 with continuation (carry includes delay_state + offset) — low-level path
# Note: exact PRNG bulk reuse across segments requires Model-level continuation (see next block); low-level re-splits key,
# so a tiny jitter can appear at low level even at noise_scale=0 due to distinct indexing paths.
# We use this to motivate the Model-level exact continuation below.
n_half = 200
V_a, S_a, src_a, st_a = simulate_edge_recurrent_izhikevich_owned_h_k_delayed(
    params, edges_delayed, n_half, DT_MS, key_c, h_k0=h0c, owner_mask=owner_c, dynamic=True, tau_k_ms=80.0, gamma_h_enabled=True, noise_scale=0.0)
V_b, S_b, src_b, st_b = simulate_edge_recurrent_izhikevich_owned_h_k_delayed(
    params, edges_delayed, n_half, DT_MS, key_c, h_k0=h0c, owner_mask=owner_c, dynamic=True, tau_k_ms=80.0, gamma_h_enabled=True, noise_scale=0.0,
    init_state=st_a, step_indices=jnp.arange(n_half, n_half*2, dtype=jnp.int32))

S_cat = _np.concatenate([_np.asarray(S_a), _np.asarray(S_b)], axis=0)
S_full_np = _np.asarray(S_full)
max_abs_diff = float(_np.max(_np.abs(S_cat - S_full_np))) if S_cat.size else 0.0
print(f"low-level segmented vs continuous max delta spikes = {max_abs_diff} (illustrative; exactness via Model-level path next)")

# Model-level exact continuation (protocol H2 verified path, bit-exact at noise_scale=0)
try:
    from jaxfne import Simulation, RuntimeConfig
    sim_half = Simulation(duration_ms=n_half*DT_MS, dt_ms=DT_MS, seed=SEED, record_sources=True, runtime=RuntimeConfig(recurrent_backend="edge_list", enable_hdp=False, hdp_params={"noise_scale": 0.0}))
    sig_a_m, state_a_m = model.simulate(sim_half, return_state=True)
    sig_b_m, state_b_m = model.simulate(sim_half, continuation=state_a_m, return_state=True)
    sig_full_m, state_full_m = model.simulate(Simulation(duration_ms=n_cont*DT_MS, dt_ms=DT_MS, seed=SEED, record_sources=True, runtime=RuntimeConfig(recurrent_backend="edge_list", enable_hdp=False, hdp_params={"noise_scale": 0.0})), return_state=True)
    cat_m = _np.concatenate([_np.asarray(sig_a_m.spikes), _np.asarray(sig_b_m.spikes)], axis=0)
    full_m = _np.asarray(sig_full_m.spikes)
    max_abs_diff_m = float(_np.max(_np.abs(cat_m - full_m)))
    print(f"Model-level continuation (delay-free, noise_scale=0): max delta spikes = {max_abs_diff_m} (expect 0, per tests/test_protocol_h_rbd_h2.py)")
    assert max_abs_diff_m == 0.0, "Model continuation should be bit-exact at noise_scale=0"
    print("H2 continuation verified: Sim_T1+T2 == Sim_T2(Sim_T1) via ContinuationState (delay_state + continuation_step_offset)")
    max_abs_diff = max_abs_diff_m
except Exception as e:
    print(f"Model-level continuation demo: {e}")


## Visualize: fading memory window & effective X

The transient while H>1 elevates excitability (b_eff) → rate bump that **fades** as H recovers. We plot mean |H−1| and rate vs time.

In [ ]:
# Use the τ=30 vs 200 runs to visualize fading
t = _np.arange(n_steps_02)*DT_MS
mean_abs_H_fast = _np.abs(_np.asarray(info_fast["H_K_trace"]).mean(axis=1)-1.0)
mean_abs_H_slow = _np.abs(_np.asarray(info_slow["H_K_trace"]).mean(axis=1)-1.0)
rate_fast = _np.asarray(_np.asarray(jtfne.emitters.simulate_edge_recurrent_izhikevich_dynamic_h_k_recovery(params, edges, n_steps_02, DT_MS, key, h_k0=h_k0_pert, tau_k_ms=30.0, noise_scale=0.0)[1]).mean(axis=1))*1000.0/DT_MS
# Actually reuse earlier S traces for rate — recompute windowed mean via convolution for smoothness
def _windowed_rate(S, win=20):
    w = _np.ones(win)/win
    return _np.convolve(_np.asarray(S).mean(axis=1)*1000.0/DT_MS, w, mode='same')
fig, axes = plt.subplots(2,1,figsize=(9,5), sharex=True)
axes[0].plot(t, mean_abs_H_fast, label="τ=30 ms"); axes[0].plot(t, mean_abs_H_slow, label="τ=200 ms"); axes[0].set_ylabel("|H−1| mean"); axes[0].legend(); axes[0].grid(alpha=0.2); axes[0].set_title("RBD fading: H deviation vs time (memory window = τ)")
axes[1].plot(t, _windowed_rate(S_dyn), label=f"run H0=2.0 τ={tau_k_ms} ms"); axes[1].set_ylabel("mean rate (Hz, windowed)"); axes[1].set_xlabel("time (ms)"); axes[1].grid(alpha=0.2)
plt.close(fig)
fig

# Optional visualize bundle on the RBD signals (build Signals wrapper to reuse panel code)
try:
    # Wrap raw arrays into a Signals-like for visualize: simulate via high-level for one window
    from jaxfne import Simulation
    sim_rbd = Simulation(duration_ms=DURATION_MS_02, dt_ms=DT_MS, seed=SEED)
    sig_rbd = model.simulate(sim_rbd)
    bund = jtfne.visualize(model, sig_rbd, backend="static")
    print(f"visualize (optional) — {len(bund.figures)} proxy panels rendered")
except Exception as e:
    print(f"visualize optional skipped: {e}")


## Configured → Realized → Effective (RBD)

Demonstrates that **timescale and continuation are effective properties** of a realized RBS+delay configuration, not new model classes.

In [ ]:
configured_rbd = {"rbd_family": "f1", "tau_h_ms": [30.0, 200.0], "kappa_h": 0.0, "beta_h": 0.2, "delay": "heterogeneous 0-2 steps (0-1 ms)"}
realized_rbd = {"n": N, "H0": float(h_k0_pert[0]), "tau_fast": 30.0, "tau_slow": 200.0, "H_end_fast_100ms": float(Hf[idx_100]), "H_end_slow_100ms": float(Hs[idx_100]), "continuation_max_abs_diff": float(max_abs_diff)}
effective_rbd = {"fading": "fast τ decays by 100 ms more than slow", "X_reflects_H": bool(diff_rbd>0), "memory_window": "τ sets half-life ~ln2·τ"}
print(json.dumps({"configured": configured_rbd, "realized": realized_rbd, "effective": effective_rbd}, indent=2))
print("configured→realized→effective (RBD): verified")


## Export

In [ ]:
REPO_ROOT = next((q for q in [Path.cwd(), *Path.cwd().parents] if (q/"jaxfne").is_dir() and (q/"pyproject.toml").exists()), Path.cwd())
OUTPUT_DIR = REPO_ROOT / "artifacts/tutorials/etudes/outputs/mechanism_02"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
manifest = {"artifact_class": "tutorial", "artifact_id": "mechanism_02_RBD_memory", "tutorial": "02_Xt_Ht1_recovery_continuation_timescales",
            "N": N, "N_canonical_reference": 1000, "rbd": configured_rbd, "realized": realized_rbd, "effective": effective_rbd,
            "demeans": "F0 null vs F1 expressed", "continuation": "delay_state+continuation_step_offset bit-exact at noise=0",
            "physical_amplitude_calibrated": False, "delta_science": 0}
Path(OUTPUT_DIR/"manifest.json").write_text(json.dumps(manifest, indent=2, default=str))
print(f"manifest -> {OUTPUT_DIR/'manifest.json'}")
print("OK mechanism 02: RBD memory, recovery, continuation, timescale sweep verified")
